In [ ]:
import pandas as pd

urls = [
    "https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_0124.csv",
    "https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_0224.csv",
    "https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_0324.csv",
    "https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_0424.csv",
    "https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_0524.csv",
    "https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_0624.csv",
    "https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_0724.csv",
    "https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_0824.csv",
    "https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_0924.csv",
    "https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_1024.csv",
    "https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_1124.csv",
    "https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_1224.csv"
]


def clasificar(codigo):

    codigo = str(codigo)

    if codigo.startswith(("61", "62", "63")):
        return "Ingreso"

    if codigo.startswith(("71", "72", "73")):
        return "Costo"

    if codigo.startswith(("81", "82", "83", "84", "85", "86", "87", "88")):
        return "Gasto"

    return "Otro"


dataframes = []

for url in urls:

    print(f"Procesando: {url}")

    archivo = url.split("/")[-1]

    mes = archivo[5:7]
    anio = "20" + archivo[7:9]

    periodo = f"{anio}-{mes}-01"

    # Leer archivo completo
    contenido = pd.read_csv(
        url,
        header=None,
        encoding="utf-8"
    )

    # Buscar fila donde aparece "Concepto"
    fila_header = None

    for i in range(len(contenido)):
        valor = str(contenido.iloc[i, 0]).strip()

        if valor == "Concepto":
            fila_header = i
            break

    if fila_header is None:
        print(f"No se encontró encabezado en {archivo}")
        continue

    # Leer nuevamente usando la fila correcta como encabezado
    df = pd.read_csv(
        url,
        header=fila_header,
        encoding="utf-8"
    )

    # Renombrar primera columna
    df.rename(
        columns={df.columns[0]: "ConceptoCompleto"},
        inplace=True
    )

    # Eliminar columna Total Instituciones
    columnas_eliminar = [
        c for c in df.columns
        if "Total Instituciones" in str(c)
    ]

    df.drop(
        columns=columnas_eliminar,
        inplace=True,
        errors="ignore"
    )

    # Eliminar filas vacías
    df = df.dropna(subset=["ConceptoCompleto"])

    # Extraer código de cuenta (2 o más dígitos)
    df["CodigoCuenta"] = (
        df["ConceptoCompleto"]
        .astype(str)
        .str.extract(r"^(\d+)")
    )

    # Mantener solo registros con código
    df = df[df["CodigoCuenta"].notna()]

    # Columnas de bancos
    columnas_bancos = [
        c for c in df.columns
        if c not in [
            "ConceptoCompleto",
            "CodigoCuenta"
        ]
    ]

    # Convertir a formato largo
    df_largo = df.melt(
        id_vars=[
            "ConceptoCompleto",
            "CodigoCuenta"
        ],
        value_vars=columnas_bancos,
        var_name="NombreBanco",
        value_name="Monto"
    )

    # Limpiar nombre banco Cuscatlán
    df_largo["NombreBanco"] = (
        df_largo["NombreBanco"]
        .astype(str)
        .str.replace(r"\s*1/$", "", regex=True)
        .str.strip()
    )

    # Limpiar concepto
    df_largo["Concepto"] = (
        df_largo["ConceptoCompleto"]
        .astype(str)
        .str.replace(r"^\d+\s*", "", regex=True)
        .str.strip()
    )

    # Clasificar cuenta
    df_largo["TipoCuenta"] = (
        df_largo["CodigoCuenta"]
        .apply(clasificar)
    )

    # Fecha
    df_largo["Periodo"] = periodo

    # Convertir monto
    df_largo["Monto"] = pd.to_numeric(
        df_largo["Monto"],
        errors="coerce"
    )

    # Eliminar montos nulos
    df_largo = df_largo[
        df_largo["Monto"].notna()
    ]

    # Seleccionar columnas finales
    df_largo = df_largo[
        [
            "Periodo",
            "NombreBanco",
            "CodigoCuenta",
            "Concepto",
            "TipoCuenta",
            "Monto"
        ]
    ]

    dataframes.append(df_largo)

# Unir todos los meses
df_final = pd.concat(
    dataframes,
    ignore_index=True
)

# Exportar CSV final
df_final.to_csv(
    "EstadoResultadosHistorico2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nPrimeras filas:")
print(df_final.head())

print("\nCantidad de registros:")
print(len(df_final))

print("\nArchivo generado: EstadoResultadosHistorico2024.csv")

Procesando: https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_0124.csv
Procesando: https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_0224.csv
Procesando: https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_0324.csv
Procesando: https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_0424.csv
Procesando: https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_0524.csv
Procesando: https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_0624.csv
Procesando: https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/heads/main/Datos/2024/er_b_0724.csv
Procesando: https://raw.githubusercontent.com/GeorginaLissethJimenezVentura/PROYECTO-BI/refs/head

**CREANDO DIMENSION BANCO**

In [ ]:
import pandas as pd

df = pd.read_csv("EstadoResultadosHistorico.csv")

dim_banco = (
    df[["NombreBanco"]]
    .drop_duplicates()
    .sort_values("NombreBanco")
    .reset_index(drop=True)
)

dim_banco["IdBanco"] = range(1, len(dim_banco) + 1)

dim_banco = dim_banco[
    ["IdBanco", "NombreBanco"]
]

dim_banco.to_csv(
    "DimBanco.csv",
    index=False,
    encoding="utf-8-sig"
)

print(dim_banco)

    IdBanco                              NombreBanco
0         1                        Banco Abank, S.A.
1         2                     Banco Agrícola, S.A.
2         3        Banco Atlantida El Salvador, S.A.
3         4          Banco Azul de El Salvador, S.A.
4         5                 Banco Cuscatlán Sv, S.A.
5         6     Banco Cuscatlán de El Salvador, S.A.
6         7       Banco Davivienda Salvadoreño, S.A.
7         8  Banco G&T Continental El Salvador, S.A.
8         9    Banco Hipotecario de El Salvador, S.A
9        10       Banco Industrial El Salvador, S.A.
10       11                    Banco Promérica, S.A.
11       12           Banco de América Central, S.A.
12       13            Banco de Fomento Agropecuario
13       14      Citibank N.A., Sucursal El Salvador


**DIMENSION CUENTA**

In [ ]:
df = pd.read_csv("EstadoResultadosHistorico.csv")

dim_cuenta = (
    df[
        ["CodigoCuenta",
         "Concepto",
         "TipoCuenta"]
    ]
    .drop_duplicates()
    .sort_values("CodigoCuenta")
)

dim_cuenta["IdCuenta"] = range(
    1,
    len(dim_cuenta) + 1
)

dim_cuenta = dim_cuenta[
    [
        "IdCuenta",
        "CodigoCuenta",
        "Concepto",
        "TipoCuenta"
    ]
]

dim_cuenta.to_csv(
    "DimCuenta.csv",
    index=False,
    encoding="utf-8-sig"
)

**DIMENSION TIEMPO**

In [ ]:


import pandas as pd

archivos = [
    "EstadoResultadosHistorico.csv",
    "EstadoResultadosHistorico2022.csv",
    "EstadoResultadosHistorico2023.csv",
    "EstadoResultadosHistorico2024.csv",
    "EstadoResultadosHistorico2025.csv"
]

df = pd.concat(
    [pd.read_csv(archivo) for archivo in archivos],
    ignore_index=True
)

print(f"Registros: {len(df)}")
print(df.head())

# Obtener fechas únicas
dim_tiempo = (
    df[["Periodo"]]
    .drop_duplicates()
    .rename(columns={"Periodo": "Fecha"})
)

# Convertir a fecha
dim_tiempo["Fecha"] = pd.to_datetime(
    dim_tiempo["Fecha"]
)

# Ordenar cronológicamente
dim_tiempo = dim_tiempo.sort_values(
    "Fecha"
).reset_index(drop=True)

# Crear IdTiempo
dim_tiempo["IdTiempo"] = range(
    1,
    len(dim_tiempo) + 1
)

# Año
dim_tiempo["Anio"] = dim_tiempo["Fecha"].dt.year

# Mes
dim_tiempo["Mes"] = dim_tiempo["Fecha"].dt.month

# Nombre del mes
meses = {
    1: "Enero",
    2: "Febrero",
    3: "Marzo",
    4: "Abril",
    5: "Mayo",
    6: "Junio",
    7: "Julio",
    8: "Agosto",
    9: "Septiembre",
    10: "Octubre",
    11: "Noviembre",
    12: "Diciembre"
}

dim_tiempo["NombreMes"] = (
    dim_tiempo["Mes"]
    .map(meses)
)

# Trimestre
dim_tiempo["Trimestre"] = (
    dim_tiempo["Fecha"]
    .dt.quarter
)

# Orden de columnas
dim_tiempo = dim_tiempo[
    [
        "IdTiempo",
        "Fecha",
        "Anio",
        "Mes",
        "NombreMes",
        "Trimestre"
    ]
]

# Exportar
dim_tiempo.to_csv(
    "DimTiempo.csv",
    index=False,
    encoding="utf-8-sig"
)

print(dim_tiempo.head())
print(f"\nRegistros: {len(dim_tiempo)}")

Registros: 24320
      Periodo           NombreBanco  CodigoCuenta  \
0  2021-01-01  Banco Agrícola, S.A.            61   
1  2021-01-01  Banco Agrícola, S.A.          6110   
2  2021-01-01  Banco Agrícola, S.A.            62   
3  2021-01-01  Banco Agrícola, S.A.          6210   
4  2021-01-01  Banco Agrícola, S.A.            63   

                                    Concepto TipoCuenta        Monto  
0  Ingresos de operaciones de intermediación    Ingreso  29190.19018  
1  Ingresos de operaciones de intermediación    Ingreso  29190.19018  
2              Ingresos de otras operaciones    Ingreso   5849.30529  
3              Ingresos de otras operaciones    Ingreso   5849.30529  
4                  Ingresos no operacionales    Ingreso   3727.63368  
   IdTiempo      Fecha  Anio  Mes NombreMes  Trimestre
0         1 2021-01-01  2021    1     Enero          1
1         2 2021-02-01  2021    2   Febrero          1
2         3 2021-03-01  2021    3     Marzo          1
3         4 2021-0

***DIMENSION HECHO**

In [ ]:
import pandas as pd

# Asegurar mismo tipo de dato
df["Periodo"] = pd.to_datetime(df["Periodo"])
dim_tiempo["Fecha"] = pd.to_datetime(dim_tiempo["Fecha"])

# Cruces para obtener las llaves sustitutas
fact = (
    df.merge(
        dim_banco,
        on="NombreBanco"
    )
    .merge(
        dim_cuenta,
        on=["CodigoCuenta", "Concepto", "TipoCuenta"]
    )
    .merge(
        dim_tiempo,
        left_on="Periodo",
        right_on="Fecha"
    )
)

# Limpiar monto
fact["Monto"] = pd.to_numeric(
    fact["Monto"],
    errors="coerce"
)

fact["Monto"] = (
    fact["Monto"]
    .fillna(0)
    .round(2)
)

# Crear IdHecho
fact["IdHecho"] = range(
    1,
    len(fact) + 1
)

# Seleccionar columnas finales
fact = fact[
    [
        "IdHecho",
        "IdBanco",
        "IdCuenta",
        "IdTiempo",
        "Monto"
    ]
]

# Exportar
fact.to_csv(
    "FactEstadoResultados.csv",
    index=False,
    encoding="utf-8-sig",
    float_format="%.2f"
)

print(fact.head())
print(f"\nRegistros: {len(fact)}")

   IdHecho  IdBanco  IdCuenta  IdTiempo     Monto
0        1        2         1         1  29190.19
1        2        2        10         1  29190.19
2        3        2         2         1   5849.31
3        4        2        11         1   5849.31
4        5        2         3         1   3727.63

Registros: 15319
